# exp 6

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from pathlib import Path
import os, re, json
import pandas as pd
from torchvision import transforms
import timm
import numpy as np
from timm.utils import AttentionExtract
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, roc_auc_score,
    average_precision_score, precision_recall_fscore_support,
    confusion_matrix, classification_report
)

import shutil

from torchvision.models.feature_extraction import create_feature_extractor, get_graph_node_names

import torch

from PIL import Image as PILImage
                
from typing import List, Dict, Tuple, Optional
from collections import defaultdict

import seaborn as sns

import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms
from scipy.ndimage import zoom, gaussian_filter
import warnings
import traceback
# Import GradCAM stuff
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image
from PIL import Image as PILImage

## Analysis of results from exp 3 to see insights from all the three models (not used in thesis)

### Check misclassified patches within their sequences

#### Patchcore

In [ ]:
# Paths & config 
CSV_PATH = "./results_experiment3/Patchcore/seed0_IM224_WR50_L2-3_P01_D1024-1024_PS-3_AN-1_S0/per_image/mvtec_fold_079/image_scores.csv"
VAL_ROOT = Path("./blurclahe_data_folds/blurclahe_inner_seed79/folds/fold_0/val")

pattern = re.compile(
    r'(?P<station>.+?)_'
    r'(?P<image_id>\d+_[0-9\.]+)_patches_patch_'
    r'(?P<x>\d+)_'
    r'(?P<y>\d+)_'
    r'(?P<angle>[0-9\.]+)_'
    r'(?P<track>-?\d+)'
    r'(?:_(?P<side>left|right))?'
    r'(?:\.[^.]+)?$'
)

def parse_seq_id(fn: str):
    m = pattern.search(os.path.basename(fn))
    if not m:
        return None
    d = m.groupdict()
    seq_id = f"{d['station']}_{d['track']}_{d.get('side','')}"
    return seq_id

def map_to_val_path(fullpath: str) -> str:
    parts = fullpath.replace("\\", "/").split("/")
    if len(parts) >= 2:
        tail = os.path.join(parts[-2], parts[-1])
    else:
        tail = parts[-1]
    return str(VAL_ROOT / tail)

# Load CSV & preprocess
df = pd.read_csv(CSV_PATH)

df["seq_id"] = df["filepath"].apply(parse_seq_id)
df = df.dropna(subset=["seq_id"]).copy()

# Determine misclassification
if "is_correct" in df.columns:
    df["is_misclassified"] = ~df["is_correct"]
else:
    df["is_misclassified"] = df["label"] != df["predicted_label"]

# Setup binary true / pred for defect class
if "is_defect" in df.columns:
    df["true_is_defect"] = df["is_defect"].astype(int)
    # Assume predicted_label is integer 0/1 for defect/good
    df["pred_is_defect"] = df["predicted_label"].astype(int)
else:
    df["true_is_defect"] = (df["label"] == "defect").astype(int)
    df["pred_is_defect"] = (df["predicted_label"] == "defect").astype(int)

# Compute confusion metrics
tp = int(((df["true_is_defect"] == 1) & (df["pred_is_defect"] == 1)).sum())
tn = int(((df["true_is_defect"] == 0) & (df["pred_is_defect"] == 0)).sum())
fp = int(((df["true_is_defect"] == 0) & (df["pred_is_defect"] == 1)).sum())
fn = int(((df["true_is_defect"] == 1) & (df["pred_is_defect"] == 0)).sum())

print(f"TP = {tp} | TN = {tn} | FP = {fp} | FN = {fn}")
print("--------------------------")

# Map to VAL_ROOT paths
df["image_path"] = df["filepath"].apply(map_to_val_path)
df["score_def"] = df["score"]

# Plot function skipping blank rows

def plot_sequence_skip_blank_rows_no_whitespace(seq_df, seq_id, patches_per_row=10):
    seq_df = seq_df.sort_values("image_path")
    if not seq_df["is_misclassified"].any():
        return

    total = len(seq_df)
    miscount = int(seq_df["is_misclassified"].sum())
    n_rows_all = (total + patches_per_row - 1) // patches_per_row

    # find which logical rows to show (those having ≥1 misclassified)
    keep_rows = []
    for r in range(n_rows_all):
        start = r * patches_per_row
        end = min((r + 1) * patches_per_row, total)
        row_slice = seq_df.iloc[start:end]
        if row_slice["is_misclassified"].any():
            keep_rows.append(r)

    if not keep_rows:
        return

    n_rows = len(keep_rows)
    fig, axes = plt.subplots(n_rows, patches_per_row,
                              figsize=(2 * patches_per_row, 2 * n_rows),
                              squeeze=False)

    fig.suptitle(f"Seq {seq_id} — total {total}, mis = {miscount}", fontsize=12)

    for plot_row_idx, r in enumerate(keep_rows):
        start = r * patches_per_row
        end = min((r + 1) * patches_per_row, total)
        for c in range(patches_per_row):
            ax = axes[plot_row_idx, c]
            idx_global = start + c
            if idx_global >= total:
                ax.axis("off")
                continue
            row = seq_df.iloc[idx_global]
            img_path = row.image_path
            if not os.path.isfile(img_path):
                ax.axis("off")
                continue
            img = Image.open(img_path).convert("RGB")
            border = "red" if row.is_misclassified else "green"
            bordered = ImageOps.expand(img, border=3, fill=border)
            ax.imshow(bordered)
            ax.axis("off")
            title = f"T:{row.label if hasattr(row, 'label') else ''}\nP:{row.predicted_label}\nScore:{row.score_def:.2f}"
            ax.set_title(title, fontsize=6)

    plt.tight_layout()
    plt.show()

# --- Run plot on each sequence with misclassifications ---
for seq_id, grp in df.groupby("seq_id"):
    plot_sequence_skip_blank_rows_no_whitespace(grp, seq_id)


#### deit-s

##### Move files with ensemble predictions

In [ ]:
base_dir = Path("./results_experiment3/DeiT-Small16")

# Copy fold prediction files
for fold in range(5):
    src = (
        base_dir
        / "blurclahe_inner_seed79"
        / f"fold_{fold}"
        / "ensemble_predictions"
        / "val_predictions_ensemble.csv"
    )

    dst = base_dir / f"fold_{fold}.csv"

    shutil.copy2(src, dst)
    print(f"Copied: {src} -> {dst}")

# Create class_mapping.json
class_mapping = {
    "class_names": [
        "defective",
        "good"
    ],
    "positive_class": "defective",
    "pos_idx": 0
}

class_mapping_path = base_dir / "class_mapping.json"

with open(class_mapping_path, "w") as f:
    json.dump(class_mapping, f, indent=2)

print(f"Created: {class_mapping_path}")

##### Analyse

In [ ]:
# Configuration 
RUN_DIR = Path("./results_experiment3/DeiT-Small16")
VAL_ROOT = Path("./blurclahe_data_folds/blurclahe_inner_seed79/folds/fold_4/val")

pattern = re.compile(
    r'(?P<station>.+?)_'
    r'(?P<image_id>\d+_[0-9\.]+)_patches_patch_'
    r'(?P<x>\d+)_'
    r'(?P<y>\d+)_'
    r'(?P<angle>[0-9\.]+)_'
    r'(?P<track>-?\d+)'
    r'(?:_(?P<side>left|right))?'
    r'(?:\.[^.]+)?$'
)

# Load predictions, metrics, mapping 
preds_path = RUN_DIR / "fold_4.csv"
val_df = pd.read_csv(preds_path)

# Attach image paths in same ordering
val_image_paths = []
for cls in ["defective", "good"]:
    cls_dir = VAL_ROOT / cls
    for img_file in sorted(cls_dir.glob("*")):
        if img_file.suffix.lower() in [".png", ".jpg", ".jpeg", ".bmp"]:
            val_image_paths.append(str(img_file))
val_df["image_path"] = val_image_paths

# with open(RUN_DIR / "val_metrics.json", "r") as f:
#     metrics = json.load(f)
with open(RUN_DIR / "class_mapping.json", "r") as f:
    mapping = json.load(f)
class_names = mapping["class_names"]

# # Print confusion metrics
# print(f"TP: {metrics['tp']} | TN: {metrics['tn']} | FP: {metrics['fp']} | FN: {metrics['fn']}")
# print("----------------")

# Parse filenames & annotate 
def parse_filename(fn):
    m = pattern.search(os.path.basename(fn))
    if not m:
        return None
    d = m.groupdict()
    d["station_track_side"] = f"{d['station']}_{d['track']}_{d.get('side','')}"
    # also capture image_id so we can sort
    d["image_id"] = d.get("image_id")
    return d

meta_df = pd.DataFrame([parse_filename(p) for p in val_df["image_path"]])
val_df = pd.concat([val_df, meta_df], axis=1)
val_df.dropna(subset=["station_track_side"], inplace=True)

val_df["is_misclassified"] = val_df["y_true"] != val_df["y_pred_ensemble"]
val_df["true_label"] = val_df["y_true"].apply(lambda x: class_names[x])
val_df["pred_label"] = val_df["y_pred_ensemble"].apply(lambda x: class_names[x])
val_df["defective_score"] = val_df["p_class_0_avg"]

# --- New plot function for Deit sequences with skip blank rows ---
def plot_sequence_skip(seq_df, seq_id, patches_per_row=10):
    # Must sort by image_id (string or numeric) so patches in correct order
    seq_df = seq_df.sort_values("image_id")
    if not seq_df["is_misclassified"].any():
        return

    total = len(seq_df)
    miscount = int(seq_df["is_misclassified"].sum())
    n_rows_all = (total + patches_per_row - 1) // patches_per_row

    # Determine which row indices to actually plot (those containing at least one mis)
    keep_rows = []
    for r in range(n_rows_all):
        start = r * patches_per_row
        end = min((r + 1) * patches_per_row, total)
        row_slice = seq_df.iloc[start:end]
        if row_slice["is_misclassified"].any():
            keep_rows.append(r)

    if not keep_rows:
        return

    n_rows = len(keep_rows)
    fig, axes = plt.subplots(n_rows, patches_per_row,
                              figsize=(2 * patches_per_row, 2 * n_rows),
                              squeeze=False)
    fig.suptitle(f"Seq {seq_id} — total {total}, mis = {miscount}", fontsize=12)

    for plot_row_idx, r in enumerate(keep_rows):
        start = r * patches_per_row
        end = min((r + 1) * patches_per_row, total)
        for c in range(patches_per_row):
            ax = axes[plot_row_idx, c]
            idx_global = start + c
            if idx_global >= total:
                ax.axis("off")
                continue
            row = seq_df.iloc[idx_global]
            img_path = row.image_path
            if not os.path.isfile(img_path):
                ax.axis("off")
                continue
            img = Image.open(img_path).convert("RGB")
            border = "red" if row.is_misclassified else "green"
            bordered = ImageOps.expand(img, border=3, fill=border)
            ax.imshow(bordered)
            ax.axis("off")
            title = f"T:{row.true_label}\nP:{row.pred_label}\nScore:{row.defective_score:.2f}"
            ax.set_title(title, fontsize=6)

    plt.tight_layout()
    plt.show()


#  Run for each sequence 
for seq_id, grp in val_df.groupby("station_track_side"):
    plot_sequence_skip(grp, seq_id)


In [ ]:

# CONFIG 
CSV_PATH = Path("./results_experiment3/DeiT-Small16/fold_4.csv")   # path to ensembled fold CSV
SAVE_CSV = True                                 # set False if you only want terminal output

# FILENAME PARSER 
pattern = re.compile(
    r'(?P<station>.+?)_(?P<image_id>\d+_[0-9\.]+)_patches_patch_'
    r'(?P<x>\d+)_(?P<y>\d+)_'
    r'(?P<angle>[0-9\.]+)_(?P<track>-?\d+)'
    r'(?:_(?P<side>left|right))?'
)

def parse_filename(fname):
    m = pattern.match(Path(fname).stem)
    if not m:
        return "?", "?", "?"
    return m["station"], m["track"], m["side"] or "na"

# LOAD DATA 
df = pd.read_csv(CSV_PATH)

# Parse filename info
df[["station", "track", "side"]] = df["filename"].apply(lambda f: pd.Series(parse_filename(f)))

# Label each prediction as TP / TN / FP / FN 
def label_case(r):
    if r.y_true == 1 and r.y_pred_ensemble == 1: return "TN"
    if r.y_true == 0 and r.y_pred_ensemble == 0: return "TP"
    if r.y_true == 1 and r.y_pred_ensemble == 0: return "FP"
    if r.y_true == 0 and r.y_pred_ensemble == 1: return "FN"
    return "?"

df["case"] = df.apply(label_case, axis=1)

# Sort by frame order 
df["frame_order"] = df["filename"].str.extract(r'_(\d+)_')[0].astype(float)
df = df.sort_values(["station", "track", "side", "frame_order"])

# PRINT PER-SEQUENCE SUMMARIES 
grouped = df.groupby(["station", "track", "side"], sort=False)
all_records = []
for (station, track, side), g in grouped:
    seq_name = f"{station}_track{track}_{side}"
    print(f"\nSequence: {seq_name} ({len(g)} patches)")
    print("-"*60)
    for _, row in g.iterrows():
        fname = Path(row.filename).name
        score = row.p_class_0_avg
        case = row.case
        print(f"-  {case:2s}  score={score:5.3f}")
        all_records.append({
            "sequence": seq_name,
            "filename": fname,
            "case": case,
            "score": score,
            "y_true": row.y_true,
            "y_pred": row.y_pred_ensemble
        })

# Optional: save full summary to CSV 
if SAVE_CSV:
    out_df = pd.DataFrame(all_records)
    out_csv = CSV_PATH.with_name(CSV_PATH.stem + "_sequence_summary.csv")
    out_df.to_csv(out_csv, index=False)
    print(f"\n Saved detailed summary to: {out_csv}")


### resnet-18

In [ ]:

# Configuration
RUN_DIR = Path("./results_experiment3/Resnet-18/blurclahe_inner_seed79/fold_0/seed0")
VAL_ROOT = Path("./blurclahe_data_folds/blurclahe_inner_seed79/folds/fold_0/val")

# RUN_DIR = Path("./blurclahe_inner_seed0/fold_1/seed0")
# VAL_ROOT = Path("./blurclahe_data_folds/blurclahe_inner_seed0/folds/fold_1/val")

pattern = re.compile(
    r'(?P<station>.+?)_'
    r'(?P<image_id>\d+_[0-9\.]+)_patches_patch_'
    r'(?P<x>\d+)_'
    r'(?P<y>\d+)_'
    r'(?P<angle>[0-9\.]+)_'
    r'(?P<track>-?\d+)'
    r'(?:_(?P<side>left|right))?'
    r'(?:\.[^.]+)?$'
)

# --- Load predictions, metrics, mapping ---
preds_path = RUN_DIR / "val_predictions.csv"
val_df = pd.read_csv(preds_path)

# Attach image paths in same ordering
val_image_paths = []
for cls in ["defective", "good"]:
    cls_dir = VAL_ROOT / cls
    for img_file in sorted(cls_dir.glob("*")):
        if img_file.suffix.lower() in [".png", ".jpg", ".jpeg", ".bmp"]:
            val_image_paths.append(str(img_file))
val_df["image_path"] = val_image_paths

with open(RUN_DIR / "val_metrics.json", "r") as f:
    metrics = json.load(f)
with open(RUN_DIR / "class_mapping.json", "r") as f:
    mapping = json.load(f)
class_names = mapping["class_names"]

# Print confusion metrics
print(f"TP: {metrics['tp']} | TN: {metrics['tn']} | FP: {metrics['fp']} | FN: {metrics['fn']}")
print("------------------------------------------------------------")

# --- Parse filenames & annotate ---
def parse_filename(fn):
    m = pattern.search(os.path.basename(fn))
    if not m:
        return None
    d = m.groupdict()
    d["station_track_side"] = f"{d['station']}_{d['track']}_{d.get('side','')}"
    # also capture image_id so we can sort
    d["image_id"] = d.get("image_id")
    return d

meta_df = pd.DataFrame([parse_filename(p) for p in val_df["image_path"]])
val_df = pd.concat([val_df, meta_df], axis=1)
val_df.dropna(subset=["station_track_side"], inplace=True)

val_df["is_misclassified"] = val_df["y_true"] != val_df["y_pred"]
val_df["true_label"] = val_df["y_true"].apply(lambda x: class_names[x])
val_df["pred_label"] = val_df["y_pred"].apply(lambda x: class_names[x])
val_df["defective_score"] = val_df["p_class_0"]

# --- New plot function for Deit sequences with skip blank rows ---
def plot_sequence_skip(seq_df, seq_id, patches_per_row=10):
    # Must sort by image_id (string or numeric) so patches in correct order
    seq_df = seq_df.sort_values("image_id")
    if not seq_df["is_misclassified"].any():
        return

    total = len(seq_df)
    miscount = int(seq_df["is_misclassified"].sum())
    n_rows_all = (total + patches_per_row - 1) // patches_per_row

    # Determine which row indices to actually plot (those containing at least one mis)
    keep_rows = []
    for r in range(n_rows_all):
        start = r * patches_per_row
        end = min((r + 1) * patches_per_row, total)
        row_slice = seq_df.iloc[start:end]
        if row_slice["is_misclassified"].any():
            keep_rows.append(r)

    if not keep_rows:
        return

    n_rows = len(keep_rows)
    fig, axes = plt.subplots(n_rows, patches_per_row,
                              figsize=(2 * patches_per_row, 2 * n_rows),
                              squeeze=False)
    fig.suptitle(f"Seq {seq_id} — total {total}, mis = {miscount}", fontsize=12)

    for plot_row_idx, r in enumerate(keep_rows):
        start = r * patches_per_row
        end = min((r + 1) * patches_per_row, total)
        for c in range(patches_per_row):
            ax = axes[plot_row_idx, c]
            idx_global = start + c
            if idx_global >= total:
                ax.axis("off")
                continue
            row = seq_df.iloc[idx_global]
            img_path = row.image_path
            if not os.path.isfile(img_path):
                ax.axis("off")
                continue
            img = Image.open(img_path).convert("RGB")
            border = "red" if row.is_misclassified else "green"
            bordered = ImageOps.expand(img, border=3, fill=border)
            ax.imshow(bordered)
            ax.axis("off")
            title = f"T:{row.true_label}\nP:{row.pred_label}\nScore:{row.defective_score:.2f}"
            ax.set_title(title, fontsize=6)

    plt.tight_layout()
    plt.show()


# --- Run for each sequence ---
for seq_id, grp in val_df.groupby("station_track_side"):
    plot_sequence_skip(grp, seq_id)


### add saliency maps to missed sequences

#### deit-s initial

In [ ]:
# --- Paths / configs ---
RUN_DIR = "./results_experiment3/DeiT-Small16/blurclahe_inner_seed79/fold_0/seed0"
MODEL_PATH = os.path.join(RUN_DIR, "artifacts", "best_model.pt")
VAL_ROOT = os.path.join("blurclahe_data_folds/blurclahe_inner_seed79", "folds", "fold_0", "val")

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# Load model
model = timm.create_model("deit_small_patch16_224", pretrained=False, num_classes=2)
state = torch.load(MODEL_PATH, map_location=device)
model.load_state_dict(state)
model.to(device)
model.eval()

# Preprocessing transform
eval_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
])

# Attention rollout function (as before)
def deit_attention_rollout(model, image_path, device="cuda", discard_ratio=0.9):
    try:
        timm.layers.set_fused_attn(False)
    except Exception:
        pass

    nodes, _ = get_graph_node_names(model)
    softmax_nodes = [n for n in nodes if "attn.softmax" in n]
    if not softmax_nodes:
        return None

    extractor = create_feature_extractor(model, return_nodes={n: n for n in softmax_nodes})

    img = Image.open(image_path).convert("RGB")
    x = eval_tf(img).unsqueeze(0).to(device)

    with torch.no_grad():
        out = extractor(x)

    attn_maps = []
    for name, att in out.items():
        att = att.squeeze(0)
        attn_maps.append(att)

    attn_avg = [a.mean(0) for a in attn_maps]

    joint = torch.eye(attn_avg[0].size(0), device=attn_avg[0].device)
    for a in attn_avg:
        a2 = a + torch.eye(a.size(0), device=a.device)
        a2 = a2 / a2.sum(dim=-1, keepdim=True)
        joint = a2 @ joint

    v = joint[0, 1:]
    num_p = v.size(0)
    g = int(np.sqrt(num_p))
    mask = v.reshape(g, g).cpu().numpy()
    mask = (mask - mask.min()) / (mask.max() - mask.min() + 1e-8)
    thresh = np.quantile(mask, discard_ratio)
    mask[mask < thresh] = 0.0

    mask_img = Image.fromarray((mask * 255).astype(np.uint8)).resize(img.size)
    overlay = Image.blend(img, ImageOps.colorize(mask_img, "black", "red"), alpha=0.4)
    return overlay

# ==Load predictions & metadata =
preds_df = pd.read_csv(os.path.join(RUN_DIR, "val_predictions.csv"))

# Build list of image paths in same ordering
all_paths = []
for cls in ["defective", "good"]:
    cls_dir = os.path.join(VAL_ROOT, cls)
    for fn in sorted(os.listdir(cls_dir)):
        if fn.lower().endswith((".png", ".jpg", ".jpeg", ".bmp")):
            all_paths.append(os.path.join(cls_dir, fn))
preds_df["image_path"] = all_paths

# Load class mapping
with open(os.path.join(RUN_DIR, "class_mapping.json"), "r") as f:
    class_mapping = json.load(f)
class_names = class_mapping["class_names"]
pos_idx = class_mapping.get("pos_idx", 0)  # positive = “defective” class index

# Parse filename info
pattern = re.compile(
    r'(?P<station>.+?)_'
    r'(?P<image_id>\d+_[0-9\.]+)_patches_patch_'
    r'(?P<x>\d+)_'
    r'(?P<y>\d+)_'
    r'(?P<angle>[0-9\.]+)_'
    r'(?P<track>-?\d+)'
    r'(?:_(?P<side>left|right))?'
    r'(?:\.[^.]+)?$'
)
def parse_fn(fn):
    m = pattern.search(os.path.basename(fn))
    if not m:
        return None
    d = m.groupdict()
    d["seq_id"] = f"{d['station']}_{d['track']}_{d.get('side','')}"
    return d

meta = [parse_fn(p) for p in preds_df["image_path"]]
preds_df = pd.concat([preds_df, pd.DataFrame(meta)], axis=1)
preds_df = preds_df.dropna(subset=["seq_id"]).copy()

preds_df["is_misclassified"] = preds_df["y_true"] != preds_df["y_pred"]
preds_df["true_label"] = preds_df["y_true"].apply(lambda x: class_names[x])
preds_df["pred_label"] = preds_df["y_pred"].apply(lambda x: class_names[x])
if "p_class_0" in preds_df.columns:
    preds_df["defective_score"] = preds_df["p_class_0"]
else:
    preds_df["defective_score"] = preds_df.filter(like="p_class_").iloc[:,0]

# Compute confusion counts
tp = int(((preds_df["y_true"] == pos_idx) & (preds_df["y_pred"] == pos_idx)).sum())
tn = int(((preds_df["y_true"] != pos_idx) & (preds_df["y_pred"] != pos_idx)).sum())
fp = int(((preds_df["y_true"] != pos_idx) & (preds_df["y_pred"] == pos_idx)).sum())
fn = int(((preds_df["y_true"] == pos_idx) & (preds_df["y_pred"] != pos_idx)).sum())
print(f"Confusion counts: TP={tp}, TN={tn}, FP={fp}, FN={fn}")

# Plot only misclassified sequences once

grouped = preds_df.groupby("seq_id")

def plot_seq_if_misclassified(seq_df, seq_id):
    # Only plot sequences that have at least one misclassified patch
    if not seq_df["is_misclassified"].any():
        return

    seq_df = seq_df.sort_values("image_id")
    n = len(seq_df)
    # Pick window around misclassification
    if n > 10:
        idx = seq_df["is_misclassified"].tolist().index(True)
        start = max(0, idx - 5)
        end = min(n, start + 10)
        sub = seq_df.iloc[start:end]
    else:
        sub = seq_df

    m = len(sub)
    fig, axs = plt.subplots(2, m, figsize=(2 * m, 4))
    fig.suptitle(f"Seq {seq_id} — miscount = {seq_df['is_misclassified'].sum()}", fontsize=12)

    for j, row in enumerate(sub.itertuples()):
        # Top: patch
        img = Image.open(row.image_path).convert("RGB")
        border = "red" if row.is_misclassified else "green"
        bordered = ImageOps.expand(img, border=4, fill=border)
        axs[0, j].imshow(bordered)
        axs[0, j].axis("off")
        axs[0, j].set_title(f"T:{row.true_label}\nP:{row.pred_label}\n{row.defective_score:.2f}", fontsize=7)

        # Bottom: overlay map
        try:
            overlay = deit_attention_rollout(model, row.image_path, device=device, discard_ratio=0.9)
            if overlay is not None:
                axs[1, j].imshow(overlay)
                axs[1, j].axis("off")
                axs[1, j].set_title("Attention", fontsize=7)
            else:
                axs[1, j].axis("off")
        except Exception as e:
            axs[1, j].axis("off")

    plt.tight_layout()
    plt.show()

# Loop over sequences, but only plot each once
for seq_id, seq_df in grouped:
    plot_seq_if_misclassified(seq_df, seq_id)


#### deit-s with heatmaps

In [ ]:

# === Configuration ===

RUN_DIR = Path("./results_experiment3/DeiT-Small16/blurclahe_inner_seed79/fold_0/seed0")
VAL_ROOT = Path("./blurclahe_data_folds/blurclahe_inner_seed79/folds/fold_0/val")

pattern = re.compile(
    r'(?P<station>.+?)_'
    r'(?P<image_id>\d+_[0-9\.]+)_patches_patch_'
    r'(?P<x>\d+)_'
    r'(?P<y>\d+)_'
    r'(?P<angle>[0-9\.]+)_'
    r'(?P<track>-?\d+)'
    r'(?:_(?P<side>left|right))?'
    r'(?:\.[^.]+)?$'
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

#  Load model & attention extractor 

# Turn off fused attention so the softmax nodes remain in the graph
try:
    timm.layers.set_fused_attn(False)
except Exception:
    pass

model = timm.create_model("deit_small_patch16_224", pretrained=False, num_classes=2)
state = torch.load(RUN_DIR / "artifacts" / "best_model.pt", map_location=device)
model.load_state_dict(state)
model.to(device)
model.eval()

attn_extractor = AttentionExtract(model, method='fx')

# Preprocessing transform (should match original model pipeline)
eval_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# === Load predictions, metrics, mapping ===

preds_path = RUN_DIR / "val_predictions.csv"
preds_df = pd.read_csv(preds_path)

all_paths = []
for cls in ["defective", "good"]:
    cls_dir = VAL_ROOT / cls
    for fn in sorted(cls_dir.glob("*")):
        if fn.suffix.lower() in [".png", ".jpg", ".jpeg", ".bmp"]:
            all_paths.append(str(fn))
preds_df["image_path"] = all_paths

with open(RUN_DIR / "val_metrics.json", "r") as f:
    metrics = json.load(f)
with open(RUN_DIR / "class_mapping.json", "r") as f:
    class_mapping = json.load(f)
class_names = class_mapping["class_names"]
pos_idx = class_mapping.get("pos_idx", 0)

print(f"Confusion counts: TP={metrics.get('tp')} | TN={metrics.get('tn')} | FP={metrics.get('fp')} | FN={metrics.get('fn')}")
print("------------------------------------------------------------")

# === Parse filenames for sequence logic ===

def parse_fn(fn: str):
    m = pattern.search(os.path.basename(fn))
    if not m:
        return None
    d = m.groupdict()
    d["seq_id"] = f"{d['station']}_{d['track']}_{d.get('side','')}"
    d["image_id"] = d.get("image_id")
    return d

meta = [parse_fn(p) for p in preds_df["image_path"]]
preds_df = pd.concat([preds_df, pd.DataFrame(meta)], axis=1)
preds_df = preds_df.dropna(subset=["seq_id"]).copy()

preds_df["is_misclassified"] = preds_df["y_true"] != preds_df["y_pred"]
preds_df["true_label"] = preds_df["y_true"].apply(lambda x: class_names[x])
preds_df["pred_label"] = preds_df["y_pred"].apply(lambda x: class_names[x])
if "p_class_0" in preds_df.columns:
    preds_df["defective_score"] = preds_df["p_class_0"]
else:
    preds_df["defective_score"] = preds_df.filter(like="p_class_").iloc[:, 0]

# === Plotting with attention overlay + sequence logic ===

def plot_sequence_with_attention(seq_df, seq_id, patches_per_row=10):
    seq_df = seq_df.sort_values("image_id")
    if not seq_df["is_misclassified"].any():
        return

    total = len(seq_df)
    miscount = int(seq_df["is_misclassified"].sum())
    n_full_rows = (total + patches_per_row - 1) // patches_per_row

    # Determine which rows to keep (those with at least one mis)
    keep_rows = []
    for r in range(n_full_rows):
        start = r * patches_per_row
        end = min((r + 1) * patches_per_row, total)
        slice_df = seq_df.iloc[start:end]
        if slice_df["is_misclassified"].any():
            keep_rows.append(r)

    if not keep_rows:
        return

    n_rows = len(keep_rows)
    patch_w = 3.5
    patch_h = 3.5
    fig, axes = plt.subplots(2 * n_rows, patches_per_row,
                            figsize=(patches_per_row * patch_w, (2 * n_rows) * patch_h),
                            squeeze=False)
    fig.suptitle(f"Seq {seq_id} — total {total}, mis = {miscount}", fontsize=16)

    for plot_idx, r in enumerate(keep_rows):
        patch_axes = axes[2 * plot_idx]
        attn_axes = axes[2 * plot_idx + 1]

        start = r * patches_per_row
        end = min((r + 1) * patches_per_row, total)

        # Plot patches row
        for c in range(patches_per_row):
            ax = patch_axes[c]
            idx = start + c
            if idx >= total:
                ax.axis("off")
                continue
            row = seq_df.iloc[idx]
            img_path = row.image_path
            if not os.path.isfile(img_path):
                ax.axis("off")
                continue
            img = Image.open(img_path).convert("RGB")
            border = "red" if row.is_misclassified else "green"
            bordered = ImageOps.expand(img, border=3, fill=border)
            ax.imshow(bordered)
            ax.set_aspect('equal')
            ax.axis("off")
            title = f"T:{row.true_label}\nP:{row.pred_label}\nScore:{row.defective_score:.3f}"
            ax.set_title(title, fontsize=14)

        # Plot attention overlay row (one overlay per patch)
        for c in range(patches_per_row):
            ax = attn_axes[c]
            idx = start + c
            if idx >= total:
                ax.axis("off")
                continue
            row = seq_df.iloc[idx]
            img_path = row.image_path
            if not os.path.isfile(img_path):
                ax.axis("off")
                continue
            # generate overlay
            try:
                img = Image.open(img_path).convert("RGB")
                x = eval_tf(img).unsqueeze(0).to(device)
                attn_maps = attn_extractor(x)

                # Debug: print available keys on first patch
                if r == keep_rows[0] and c == 0:
                    print("Attention keys:", list(attn_maps.keys()))

                # choose a key (prefer last block softmax)
                key = None
                for k in attn_maps.keys():
                    if "blocks.11.attn.softmax" in k:
                        key = k
                        break
                if key is None:
                    # fallback
                    key = list(attn_maps.keys())[-1]

                att = attn_maps[key]  # shape (1, heads, tokens, tokens)
                cls_to_p = att[:, :, 0, 1:]  # (1, heads, num_patches)
                att_avg = cls_to_p.mean(dim=1).squeeze(0)  # (num_patches,)
                num_p = att_avg.size(0)
                g = int(np.sqrt(num_p))
                mask = att_avg.reshape(g, g).cpu().detach().numpy()
                mask = (mask - mask.min()) / (mask.max() - mask.min() + 1e-8)

                # Resize mask to image size
                mask_img = Image.fromarray((mask * 255).astype(np.uint8)).resize(img.size)
                overlay = Image.blend(img, ImageOps.colorize(mask_img, "black", "red"), alpha=0.4)
                ax.imshow(overlay)
                ax.axis("off")
            except Exception as e:
                print("Attention overlay error:", e, img_path)
                ax.axis("off")

    plt.tight_layout()
    plt.show()


# Run for each sequence with misclassifications
for seq_id, grp in preds_df.groupby("seq_id"):
    plot_sequence_with_attention(grp, seq_id)


#### resnet with heatmaps

In [ ]:


# === Configuration ===
RUN_DIR = Path("./results_experiment3/Resnet-18/blurclahe_inner_seed79/fold_0/seed0")
VAL_ROOT = Path("./blurclahe_data_folds/blurclahe_inner_seed79/folds/fold_0/val")

pattern = re.compile(
    r'(?P<station>.+?)_'
    r'(?P<image_id>\d+_[0-9\.]+)_patches_patch_'
    r'(?P<x>\d+)_'
    r'(?P<y>\d+)_'
    r'(?P<angle>[0-9\.]+)_'
    r'(?P<track>-?\d+)'
    r'(?:_(?P<side>left|right))?'
    r'(?:\.[^.]+)?$'
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# === Load ResNet18 model ===
model = timm.create_model("resnet18", pretrained=False, num_classes=2)
state = torch.load(RUN_DIR / "artifacts" / "best_model.pt", map_location=device)
model.load_state_dict(state)
model.to(device)
model.eval()

# Choose CAM target layer
# For timm ResNet18, last conv might be model.layer4[-1].conv2 or model.layer4[-1]
target_layer = model.layer4[-1].conv2

# Instantiate GradCAM (without use_cuda argument)
cam = GradCAM(model=model, target_layers=[target_layer])

# Preprocessing transform
eval_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# === Load predictions & attach paths ===
preds_df = pd.read_csv(RUN_DIR / "val_predictions.csv")

# Build list of patch image paths in the same ordering as preds_df
all_paths = []
for cls in ["defective", "good"]:
    cls_dir = VAL_ROOT / cls
    for fn in sorted(cls_dir.glob("*")):
        if fn.suffix.lower() in [".png", ".jpg", ".jpeg", ".bmp"]:
            all_paths.append(str(fn))
preds_df["image_path"] = all_paths  # <-- this must come before parsing seq_id

# === Load metrics & mapping ===
with open(RUN_DIR / "val_metrics.json", "r") as f:
    metrics = json.load(f)
with open(RUN_DIR / "class_mapping.json", "r") as f:
    class_mapping = json.load(f)
class_names = class_mapping["class_names"]
pos_idx = class_mapping.get("pos_idx", 0)

print(f"TP: {metrics.get('tp')} | TN: {metrics.get('tn')} | FP: {metrics.get('fp')} | FN: {metrics.get('fn')}")
print("------------------------------------------------------------")

# === Parse seq_id from filename ===
def parse_fn(fn: str):
    m = pattern.search(os.path.basename(fn))
    if not m:
        return None
    d = m.groupdict()
    d["seq_id"] = f"{d['station']}_{d['track']}_{d.get('side','')}"
    d["image_id"] = d.get("image_id")
    return d

meta = [parse_fn(p) for p in preds_df["image_path"]]
preds_df = pd.concat([preds_df, pd.DataFrame(meta)], axis=1)

# Drop rows where seq_id parsing failed
preds_df = preds_df.dropna(subset=["seq_id"]).copy()

# Add classification info
preds_df["is_misclassified"] = preds_df["y_true"] != preds_df["y_pred"]
preds_df["true_label"] = preds_df["y_true"].apply(lambda x: class_names[x])
preds_df["pred_label"] = preds_df["y_pred"].apply(lambda x: class_names[x])
if "p_class_0" in preds_df.columns:
    preds_df["defective_score"] = preds_df["p_class_0"]
else:
    preds_df["defective_score"] = preds_df.filter(like="p_class_").iloc[:, 0]

# === Plot function using CAM + sequence layout ===
def plot_seq_with_cam(seq_df, seq_id, patches_per_row=10):
    seq_df = seq_df.sort_values("image_id")
    if not seq_df["is_misclassified"].any():
        return

    total = len(seq_df)
    miscount = int(seq_df["is_misclassified"].sum())
    n_full_rows = (total + patches_per_row - 1) // patches_per_row

    keep_rows = []
    for r in range(n_full_rows):
        start = r * patches_per_row
        end = min((r + 1) * patches_per_row, total)
        if seq_df.iloc[start:end]["is_misclassified"].any():
            keep_rows.append(r)
    if not keep_rows:
        return

    n_rows = len(keep_rows)
    patch_w = 3.0
    patch_h = 3.0
    fig, axes = plt.subplots(2 * n_rows, patches_per_row,
                              figsize=(patches_per_row * patch_w, (2 * n_rows) * patch_h),
                              squeeze=False)
    fig.suptitle(f"Seq {seq_id} — total {total}, mis = {miscount}", fontsize=12)

    for plot_i, r in enumerate(keep_rows):
        patch_axes = axes[2 * plot_i]
        cam_axes = axes[2 * plot_i + 1]

        start = r * patches_per_row
        end = min((r + 1) * patches_per_row, total)

        # Patch images row
        for c in range(patches_per_row):
            ax = patch_axes[c]
            idx = start + c
            if idx >= total:
                ax.axis("off")
                continue
            row = seq_df.iloc[idx]
            img_path = row.image_path
            if not os.path.isfile(img_path):
                ax.axis("off")
                continue
            img = Image.open(img_path).convert("RGB")
            border = "red" if row.is_misclassified else "green"
            bordered = ImageOps.expand(img, border=3, fill=border)
            ax.imshow(bordered)
            ax.set_aspect('equal')
            ax.axis("off")
            title = f"T:{row.true_label}\nP:{row.pred_label}\nScore:{row.defective_score:.3f}"
            ax.set_title(title, fontsize=6)

        # CAM overlay row
        for c in range(patches_per_row):
            ax = cam_axes[c]
            idx = start + c
            if idx >= total:
                ax.axis("off")
                continue
            row = seq_df.iloc[idx]
            img_path = row.image_path
            if not os.path.isfile(img_path):
                ax.axis("off")
                continue
            try:
                pil = Image.open(img_path).convert("RGB")
                img_tensor = eval_tf(pil).unsqueeze(0).to(device)
                target_class = row.y_pred
                targets = [ClassifierOutputTarget(target_class)]
                grayscale_cam = cam(input_tensor=img_tensor, targets=targets)[0]

                # Resize CAM mask to patch size
                w, h = pil.size  # (width, height)
                # Option A: resize the heatmap (grayscale) then overlay
                # use PIL or OpenCV
                
                cam_mask = PILImage.fromarray((grayscale_cam * 255).astype('uint8'))
                cam_mask = cam_mask.resize((w, h), resample=PILImage.BILINEAR)
                cam_mask_np = np.array(cam_mask).astype(float) / 255.0

                # Now overlay
                img_np = np.array(pil).astype(float) / 255.0
                cam_overlay = show_cam_on_image(img_np, cam_mask_np, use_rgb=True)

                ax.imshow(cam_overlay)
                ax.axis("off")
            except Exception as e:
                print("CAM overlay error:", e, img_path)
                ax.axis("off")

    plt.tight_layout()
    plt.show()


# === Run plotting for each sequence that has misclassifications ===
for seq_id, grp in preds_df.groupby("seq_id"):
    plot_seq_with_cam(grp, seq_id)


# XAI Analysis of Final Model Results from experiment 5 (used in thesis)

In [ ]:

# Complete Experiment 6: XAI Analysis

# Set style
plt.style.use('seaborn-v0_8-paper')
sns.set_palette("colorblind")

# CONFIGURATION

EXP5_DIR = Path("./results_experiment5_trainbestparams")
MODEL_PATH = EXP5_DIR / "final_model_seed42.pt"
TEST_PREDICTIONS = EXP5_DIR / "test_predictions_ensemble.csv"
TEST_PREDICTIONS_MULTISCALE = EXP5_DIR / "test_predictions_ensemble_multiscale.csv"
TEST_PREDICTIONS_CONF_ISO = EXP5_DIR / "test_predictions_ensemble_conf_weighted_iso.csv"
TEST_IMAGES_DIR = Path("./blurclahe_data_folds/blurclahe_inner_seed79_train_test/test")

OUTPUT_DIR = Path("./results_experiment6_xai_NEW")
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
IMG_SIZE = 224

FILENAME_PATTERN = re.compile(
    r'(?P<station>.+?)_'
    r'(?P<image_id>\d+_[0-9\.]+)_patches_patch_'
    r'(?P<x>\d+)_'
    r'(?P<y>\d+)_'
    r'(?P<angle>[0-9\.]+)_'
    r'(?P<track>-?\d+)'
    r'(?:_(?P<side>left|right))?'
    r'(?:\.[^.]+)?$'
)

# UTILITIES

def print_section(title: str):
    print("\n" + "-"*10)
    print(f"  {title}")
    print("-"*10)

def parse_filename(filename: str) -> Dict:
    m = FILENAME_PATTERN.match(Path(filename).name)
    return m.groupdict() if m else {}

def extract_frame_number(image_id: str) -> int:
    try:
        return int(str(image_id).split('_')[0])
    except:
        return 0

# XAI METHODS 

class GradientSaliency:
    """Simple gradient-based saliency - WORKS WELL!"""
    
    def __init__(self, model: nn.Module):
        self.model = model
    
    def generate(self, img_tensor: torch.Tensor, target_class: int) -> np.ndarray:
        self.model.eval()
        img_tensor = img_tensor.clone().requires_grad_(True)
        
        output = self.model(img_tensor)
        self.model.zero_grad()
        
        score = output[0, target_class]
        score.backward()
        
        grads = img_tensor.grad.squeeze(0).abs().max(dim=0)[0].cpu().numpy()
        grads = (grads - grads.min()) / (grads.max() - grads.min() + 1e-8)
        
        return grads


class IntegratedGradients:
    """Integrated Gradients - More robust"""
    
    def __init__(self, model: nn.Module):
        self.model = model
    
    def generate(self, img_tensor: torch.Tensor, target_class: int, steps: int = 50) -> np.ndarray:
        self.model.eval()
        baseline = torch.zeros_like(img_tensor)
        integrated_grads = torch.zeros_like(img_tensor)
        
        for i in range(steps):
            alpha = i / (steps - 1)
            interpolated = baseline + alpha * (img_tensor - baseline)
            interpolated = interpolated.clone().detach().requires_grad_(True)
            
            output = self.model(interpolated)
            score = output[0, target_class]
            
            self.model.zero_grad()
            score.backward()
            
            integrated_grads += interpolated.grad
        
        integrated_grads = integrated_grads / steps
        attributions = (img_tensor - baseline) * integrated_grads
        attributions = attributions.squeeze(0).abs().max(dim=0)[0].cpu().numpy()
        
        if attributions.max() > 0:
            attributions = (attributions - attributions.min()) / (attributions.max() - attributions.min() + 1e-8)
        
        return attributions


class SmoothGrad:
    """SmoothGrad - Noise-reduced version"""
    
    def __init__(self, model: nn.Module):
        self.model = model
        self.base_saliency = GradientSaliency(model)
    
    def generate(self, img_tensor: torch.Tensor, target_class: int,
                n_samples: int = 50, noise_level: float = 0.15) -> np.ndarray:
        accumulated = torch.zeros(img_tensor.shape[2:]).to(img_tensor.device)
        
        for _ in range(n_samples):
            noise = torch.randn_like(img_tensor) * noise_level
            noisy_img = img_tensor + noise
            saliency = self.base_saliency.generate(noisy_img, target_class)
            accumulated += torch.from_numpy(saliency).to(img_tensor.device)
        
        smooth = (accumulated / n_samples).cpu().numpy()
        smooth = (smooth - smooth.min()) / (smooth.max() - smooth.min() + 1e-8)
        
        return smooth

# VISUALIZATION

def visualize_explanation(img: Image.Image, explanation: np.ndarray,
                         save_path: Path, title: str = "Saliency Map"):
    """Create publication-quality visualization."""
    
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    
    # Resize explanation
    h, w = img.size[1], img.size[0]
    exp_h, exp_w = explanation.shape
    explanation_resized = zoom(explanation, (h/exp_h, w/exp_w), order=3)
    
    # Original
    axes[0].imshow(img)
    axes[0].set_title("Original Image", fontsize=20)
    axes[0].axis('off')
    
    # Heatmap
    im1 = axes[1].imshow(explanation_resized, cmap='jet', vmin=0, vmax=1)
    axes[1].set_title("Saliency Heatmap", fontsize=20)
    axes[1].axis('off')
    plt.colorbar(im1, ax=axes[1], fraction=0.046)
    
    # Overlay
    axes[2].imshow(img, alpha=0.7)
    axes[2].imshow(explanation_resized, cmap='jet', alpha=0.5, vmin=0, vmax=1)
    axes[2].set_title("Overlay", fontsize=20)
    axes[2].axis('off')
    
    # Threshold
    threshold = 0.6
    exp_thresh = np.where(explanation_resized > threshold, explanation_resized, 0)
    axes[3].imshow(img, alpha=0.7)
    axes[3].imshow(exp_thresh, cmap='jet', alpha=0.6, vmin=0, vmax=1)
    axes[3].set_title(f"High Importance (>{threshold})", fontsize=16)
    axes[3].axis('off')
    
    plt.suptitle(title, fontsize=20, y=0.98)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"  ✓ Saved: {save_path.name}")


def compare_methods_sidebyside(img: Image.Image, saliency: np.ndarray,
                               integrated_grads: np.ndarray, smoothgrad: np.ndarray,
                               save_path: Path, title: str = "Method Comparison"):
    """Compare all methods side-by-side."""
    
    fig, axes = plt.subplots(2, 4, figsize=(20, 10))
    
    h, w = img.size[1], img.size[0]
    methods = [
        ("Gradient Saliency", saliency),
        ("Integrated Gradients", integrated_grads),
        ("SmoothGrad", smoothgrad)
    ]
    
    # Row 1: Original + Heatmaps
    axes[0, 0].imshow(img)
    axes[0, 0].set_title("Original Image", fontsize=16)
    axes[0, 0].axis('off')
    
    for i, (method_name, explanation) in enumerate(methods, 1):
        exp_h, exp_w = explanation.shape
        exp_resized = zoom(explanation, (h/exp_h, w/exp_w), order=3)
        
        im = axes[0, i].imshow(exp_resized, cmap='jet', vmin=0, vmax=1)
        axes[0, i].set_title(method_name, fontsize=16)
        axes[0, i].axis('off')
        plt.colorbar(im, ax=axes[0, i], fraction=0.046)
    
    # Row 2: Overlays
    axes[1, 0].imshow(img)
    axes[1, 0].set_title("Original (Reference)", fontsize=16)
    axes[1, 0].axis('off')
    
    for i, (method_name, explanation) in enumerate(methods, 1):
        exp_h, exp_w = explanation.shape
        exp_resized = zoom(explanation, (h/exp_h, w/exp_w), order=3)
        
        axes[1, i].imshow(img, alpha=0.6)
        axes[1, i].imshow(exp_resized, cmap='jet', alpha=0.5, vmin=0, vmax=1)
        axes[1, i].set_title(f"{method_name} Overlay", fontsize=16)
        axes[1, i].axis('off')
    
    plt.suptitle(title, fontsize=30, y=0.995)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"  ✓ Saved comparison: {save_path.name}")

# ERROR ANALYSIS

def perform_error_analysis(predictions_df: pd.DataFrame, test_images_dir: Path) -> pd.DataFrame:
    """Categorize predictions into TP, TN, FP, FN."""
    
    predictions_df['y_true_defective'] = (predictions_df['y_true'] == 0).astype(int)
    predictions_df['y_pred_defective'] = (predictions_df['y_pred'] == 0).astype(int)
    predictions_df['p_defective'] = predictions_df['p_class_0']
    
    def categorize(row):
        if row['y_true_defective'] == 1 and row['y_pred_defective'] == 1:
            return 'TP'
        elif row['y_true_defective'] == 0 and row['y_pred_defective'] == 0:
            return 'TN'
        elif row['y_true_defective'] == 0 and row['y_pred_defective'] == 1:
            return 'FP'
        else:
            return 'FN'
    
    predictions_df['category'] = predictions_df.apply(categorize, axis=1)
    
    def find_image_path(filename):
        for subfolder in ['defective', 'good']:
            path = test_images_dir / subfolder / filename
            if path.exists():
                return path
        return None
    
    predictions_df['image_path'] = predictions_df['filename'].apply(find_image_path)
    
    print_section("Error Analysis Summary")
    summary = predictions_df['category'].value_counts()
    for cat in ['TP', 'TN', 'FP', 'FN']:
        count = summary.get(cat, 0)
        print(f"  {cat}: {count:3d} ({count/len(predictions_df)*100:5.2f}%)")
    
    return predictions_df


def create_error_analysis_table(predictions_df: pd.DataFrame, output_dir: Path, n_examples: int = 5):
    """Create table with example cases."""
    
    for category in ['TP', 'TN', 'FP', 'FN']:
        cat_df = predictions_df[predictions_df['category'] == category]
        
        if len(cat_df) == 0:
            continue
        
        if category in ['TP', 'TN']:
            cat_df = cat_df.sort_values('p_defective', ascending=(category=='TN'))
        else:
            cat_df = cat_df.sort_values('p_defective', ascending=(category=='FN'))
        
        examples = cat_df.head(n_examples)[['filename', 'y_true', 'y_pred', 
                                            'p_defective', 'p_class_0', 'p_class_1']]
        examples.to_csv(output_dir / f"error_analysis_{category}.csv", index=False)
        print(f"  ✓ Saved {category} examples: {len(examples)} cases")


def create_thesis_visualizations(predictions_df, model, test_images_dir, 
                                output_dir, device='cuda', n_per_category=3):
    """Create all XAI visualizations using working methods."""
    
    print_section("Creating Thesis-Quality XAI Visualizations")
    
    # Initialize methods
    grad_sal = GradientSaliency(model)
    int_grad = IntegratedGradients(model)
    smooth_grad = SmoothGrad(model)
    
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    
    # Create subdirectories
    (output_dir / "gradient_saliency").mkdir(exist_ok=True)
    (output_dir / "integrated_gradients").mkdir(exist_ok=True)
    (output_dir / "smoothgrad").mkdir(exist_ok=True)
    (output_dir / "comparisons").mkdir(exist_ok=True)
    
    # Process each category
    for category in ['TP', 'TN', 'FP', 'FN']:
        cat_df = predictions_df[predictions_df['category'] == category]
        
        # Replace lines 294-300 with:
        
        if len(cat_df) == 0:
            print(f"\n  No {category} examples found")
            continue
        
        print(f"\n  Processing {category} examples...")
        
        if category == 'TP':
            examples = cat_df.nlargest(n_per_category, 'p_defective')  # Highest confidence defects
        elif category == 'TN':  
            examples = cat_df.nsmallest(n_per_category, 'p_defective')  # Lowest confidence defects (highest confidence normals)
        elif category == 'FP':
            examples = cat_df.nlargest(n_per_category, 'p_defective')  # Highest confidence false alarms
        else:  # FN
            examples = cat_df.nsmallest(n_per_category, 'p_defective')  # Lowest confidence misses
    
        for idx, (_, row) in enumerate(examples.iterrows(), 1):
            if row['image_path'] is None or not Path(row['image_path']).exists():
                continue
            
            img = Image.open(row['image_path']).convert('RGB')
            img_tensor = transform(img).unsqueeze(0).to(device)
            
            pred_class = row['y_pred']
            true_class = row['y_true']
            confidence = row['p_defective']
            
            class_name = {0: 'Defective', 1: 'Good'}
            title_base = (f"{category} Example : "
                         f"True={class_name[true_class]}, "
                         f"Pred={class_name[pred_class]} "
                         f"(conf={confidence:.3f})")
            
            print(f"    Processing {category} example {idx}...")
            
            # Generate explanations
            sal = grad_sal.generate(img_tensor, pred_class)
            ig = int_grad.generate(img_tensor, pred_class, steps=30)
            sg = smooth_grad.generate(img_tensor, pred_class, n_samples=30)
            
            # Save individual visualizations
            visualize_explanation(
                img, sal,
                output_dir / "gradient_saliency" / f"{category}_ex{idx}.png",
                title=f"{title_base}\nGradient Saliency"
            )
            
            visualize_explanation(
                img, ig,
                output_dir / "integrated_gradients" / f"{category}_ex{idx}.png",
                title=f"{title_base}\nIntegrated Gradients"
            )
            
            visualize_explanation(
                img, sg,
                output_dir / "smoothgrad" / f"{category}_ex{idx}.png",
                title=f"{title_base}\nSmoothGrad"
            )
            
            # Save comparison
            compare_methods_sidebyside(
                img, sal, ig, sg,
                output_dir / "comparisons" / f"{category}_ex{idx}_comparison.png",
                title=title_base
            )
    
    print("\n" + "-"*10)
    print("XAI VISUALIZATIONS COMPLETE!")
    print("-"*10)

# TEMPORAL & CONFIDENCE ANALYSIS

def analyze_temporal_sequences(predictions_df: pd.DataFrame, output_dir: Path):
    """Analyze temporal sequences."""
    
    print_section("Temporal Sequence Analysis")
    
    predictions_df['parsed'] = predictions_df['filename'].apply(parse_filename)
    predictions_df['station'] = predictions_df['parsed'].apply(lambda x: x.get('station'))
    predictions_df['track'] = predictions_df['parsed'].apply(lambda x: x.get('track'))
    predictions_df['side'] = predictions_df['parsed'].apply(lambda x: x.get('side'))
    predictions_df['image_id'] = predictions_df['parsed'].apply(lambda x: x.get('image_id'))
    predictions_df['frame_num'] = predictions_df['image_id'].apply(extract_frame_number)
    
    sequences = []
    for (station, track, side), grp in predictions_df.groupby(['station', 'track', 'side'], dropna=False):
        if len(grp) < 5:
            continue
        
        grp_sorted = grp.sort_values('frame_num')
        has_defects = grp_sorted['y_true_defective'].sum() > 0
        
        sequences.append({
            'station': station,
            'track': track,
            'side': side,
            'length': len(grp_sorted),
            'n_defects': int(grp_sorted['y_true_defective'].sum()),
            'has_defects': has_defects,
            'data': grp_sorted
        })
    
    print(f"  Found {len(sequences)} sequences")
    print(f"    With defects: {sum(s['has_defects'] for s in sequences)}")
    
    # Visualize one sequence
    defective_seqs = [s for s in sequences if s['has_defects'] and s['length'] >= 10]
    if defective_seqs:
        seq = defective_seqs[0]
        visualize_sequence(seq, output_dir)
    
    return sequences


def visualize_sequence(sequence: Dict, output_dir: Path):
    """Visualize predictions across temporal sequence."""
    
    data = sequence['data']
    fig, axes = plt.subplots(3, 1, figsize=(15, 10), sharex=True)
    frames = data['frame_num'].values
    
    # Probabilities
    axes[0].plot(frames, data['p_defective'], 'o-', label='P(Defective)', linewidth=2, markersize=6)
    axes[0].axhline(y=0.5, color='r', linestyle='--', alpha=0.5, label='Threshold')
    axes[0].fill_between(frames, 0, 1, where=data['y_true_defective']==1, 
                         alpha=0.2, color='red', label='True Defective')
    axes[0].set_ylabel('Probability', fontsize=16)
    axes[0].set_ylim([-0.05, 1.05])
    axes[0].legend(loc='upper right')
    axes[0].grid(True, alpha=0.3)
    axes[0].set_title(f"Sequence: {sequence['station']}, Track {sequence['track']}, Side {sequence['side']}", 
                     fontsize=20)
    
    # Predictions vs Ground Truth
    axes[1].scatter(frames, data['y_true_defective'], marker='s', s=100, c='green', alpha=0.7, label='Ground Truth')
    axes[1].scatter(frames, data['y_pred_defective'], marker='o', s=80, c='blue', alpha=0.7, label='Prediction')
    axes[1].set_ylabel('Class', fontsize=16)
    axes[1].set_yticks([0, 1])
    axes[1].set_yticklabels(['Good', 'Defective'])
    axes[1].legend(loc='upper right')
    axes[1].grid(True, alpha=0.3, axis='x')
    
    # Errors
    errors = (data['y_true_defective'] != data['y_pred_defective']).astype(int)
    axes[2].bar(frames, errors, color='red', alpha=0.7)
    axes[2].set_ylabel('Prediction Error', fontsize=16)
    axes[2].set_xlabel('Frame Number', fontsize=16)
    axes[2].set_yticks([0, 1])
    axes[2].set_yticklabels(['Correct', 'Error'])
    axes[2].grid(True, alpha=0.3, axis='x')
    
    plt.tight_layout()
    plt.savefig(output_dir / "sequence_temporal_analysis.png", dpi=300, bbox_inches='tight')
    plt.close()
    print(f"  ✓ Saved temporal sequence visualization")


def analyze_confidence_distribution(predictions_df: pd.DataFrame, output_dir: Path):
    """Analyze confidence distributions."""
    
    print_section("Confidence Distribution Analysis")
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 7))
    
    correct = predictions_df[predictions_df['y_true'] == predictions_df['y_pred']]
    incorrect = predictions_df[predictions_df['y_true'] != predictions_df['y_pred']]
    
    # Overall distribution
    axes[0, 0].hist(correct['p_defective'], bins=30, alpha=0.7, label=f'Correct (n={len(correct)})', color='green')
    axes[0, 0].hist(incorrect['p_defective'], bins=30, alpha=0.7, label=f'Incorrect (n={len(incorrect)})', color='red')
    axes[0, 0].axvline(x=0.5, color='black', linestyle='--', alpha=0.5)
    axes[0, 0].set_xlabel('P(Defective)', fontsize=16)
    axes[0, 0].set_ylabel('Count', fontsize=16)
    axes[0, 0].set_title('Confidence: Correct vs Incorrect', fontsize=20)
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # By category
    for cat, color in [('TP', 'darkgreen'), ('TN', 'lightgreen'), ('FP', 'orange'), ('FN', 'darkred')]:
        cat_data = predictions_df[predictions_df['category'] == cat]
        if len(cat_data) > 0:
            axes[0, 1].hist(cat_data['p_defective'], bins=20, alpha=0.6, 
                           label=f'{cat} (n={len(cat_data)})', color=color)
    
    axes[0, 1].axvline(x=0.5, color='black', linestyle='--', alpha=0.5)
    axes[0, 1].set_xlabel('P(Defective)', fontsize=16)
    axes[0, 1].set_ylabel('Count', fontsize=16)
    axes[0, 1].set_title('Confidence by Category', fontsize=20)
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    
    # Box plots
    box_data = [predictions_df[predictions_df['category'] == cat]['p_defective'].values 
                for cat in ['TP', 'TN', 'FP', 'FN']]
    bp = axes[1, 0].boxplot(box_data, labels=['TP', 'TN', 'FP', 'FN'], patch_artist=True)
    colors = ['darkgreen', 'lightgreen', 'orange', 'darkred']
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.6)
    axes[1, 0].axhline(y=0.5, color='black', linestyle='--', alpha=0.5)
    axes[1, 0].set_ylabel('P(Defective)', fontsize=16)
    axes[1, 0].set_title('Confidence by Category (Box Plot)', fontsize=20)
    axes[1, 0].grid(True, alpha=0.3, axis='y')
    
    # Scatter
    for cat, marker, color in [('TP', 'o', 'darkgreen'), ('TN', 's', 'lightgreen'),
                               ('FP', '^', 'orange'), ('FN', 'v', 'darkred')]:
        cat_data = predictions_df[predictions_df['category'] == cat]
        if len(cat_data) > 0:
            axes[1, 1].scatter(range(len(cat_data)), cat_data['p_defective'].values,
                             marker=marker, s=60, alpha=0.6, label=f'{cat} (n={len(cat_data)})', color=color)
    
    axes[1, 1].axhline(y=0.5, color='black', linestyle='--', alpha=0.5)
    axes[1, 1].set_xlabel('Sample Index', fontsize=16)
    axes[1, 1].set_ylabel('P(Defective)', fontsize=16)
    axes[1, 1].set_title('Confidence by Sample', fontsize=20)
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(output_dir / "confidence_distribution_analysis.png", dpi=300, bbox_inches='tight')
    plt.close()
    print(f"  ✓ Saved confidence analysis")

# POST-PROCESSING IMPACT

def compare_postprocessing_impact(pred_baseline: pd.DataFrame, pred_multiscale: pd.DataFrame,
                                 pred_conf_iso: pd.DataFrame, output_dir: Path):
    """Compare predictions before/after post-processing."""
    
    print_section("Post-Processing Impact Analysis")
    
    df = pred_baseline[['filename', 'y_true', 'y_pred', 'p_defective']].copy()
    df.columns = ['filename', 'y_true', 'pred_baseline', 'prob_baseline']
    
    df = df.merge(
        pred_multiscale[['filename', 'y_pred', 'p_defective']].rename(
            columns={'y_pred': 'pred_multiscale', 'p_defective': 'prob_multiscale'}
        ), on='filename'
    )
    
    df = df.merge(
        pred_conf_iso[['filename', 'y_pred']].rename(columns={'y_pred': 'pred_conf_iso'}),
        on='filename'
    )
    
    df['changed_multiscale'] = (df['pred_baseline'] != df['pred_multiscale']).astype(int)
    df['changed_conf_iso'] = (df['pred_baseline'] != df['pred_conf_iso']).astype(int)
    
    def categorize_change(row, method):
        if row[f'changed_{method}'] == 0:
            return 'No Change'
        baseline_correct = row['y_true'] == row['pred_baseline']
        new_correct = row['y_true'] == row[f'pred_{method}']
        if baseline_correct and not new_correct:
            return 'Correct → Wrong'
        elif not baseline_correct and new_correct:
            return 'Wrong → Correct'
        else:
            return 'Wrong → Wrong'
    
    df['change_type_multiscale'] = df.apply(lambda r: categorize_change(r, 'multiscale'), axis=1)
    df['change_type_conf_iso'] = df.apply(lambda r: categorize_change(r, 'conf_iso'), axis=1)
    
    print(f"\n  Multiscale: {df['changed_multiscale'].sum()}/{len(df)} changed")
    print(f"  Conf+Iso: {df['changed_conf_iso'].sum()}/{len(df)} changed")
    
    df.to_csv(output_dir / "postprocessing_comparison_detailed.csv", index=False)
    
    # Visualize
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    change_counts_multi = df['change_type_multiscale'].value_counts()
    axes[0].bar(range(len(change_counts_multi)), change_counts_multi.values, 
               color=['green', 'blue', 'red', 'orange'])
    axes[0].set_xticks(range(len(change_counts_multi)))
    axes[0].set_xticklabels(change_counts_multi.index, rotation=45, ha='right')
    axes[0].set_ylabel('Count', fontsize=16)
    axes[0].set_title('Multiscale Impact', fontsize=16)
    axes[0].grid(True, alpha=0.3, axis='y')
    
    change_counts_conf = df['change_type_conf_iso'].value_counts()
    axes[1].bar(range(len(change_counts_conf)), change_counts_conf.values, 
               color=['green', 'blue', 'red', 'orange'])
    axes[1].set_xticks(range(len(change_counts_conf)))
    axes[1].set_xticklabels(change_counts_conf.index, rotation=45, ha='right')
    axes[1].set_ylabel('Count', fontsize=16)
    axes[1].set_title('Conf+Iso Impact', fontsize=16)
    axes[1].grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.savefig(output_dir / "postprocessing_impact.png", dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"  ✓ Saved impact analysis")
    return df

# COMPREHENSIVE REPORT

def generate_xai_report(predictions_df: pd.DataFrame, comparison_df: pd.DataFrame,
                       sequences: List[Dict], output_dir: Path):
    """Generate comprehensive XAI report."""
    
    print_section("Generating XAI Report")
    
    tp = (predictions_df['category'] == 'TP').sum()
    fp = (predictions_df['category'] == 'FP').sum()
    precision = float(tp / (tp + fp)) if (tp + fp) > 0 else None
    
    report = {
        'dataset_statistics': {
            'total_samples': len(predictions_df),
            'defective_samples': int(predictions_df['y_true_defective'].sum()),
            'good_samples': int((1 - predictions_df['y_true_defective']).sum()),
        },
        'performance_summary': {
            'tp': int((predictions_df['category'] == 'TP').sum()),
            'tn': int((predictions_df['category'] == 'TN').sum()),
            'fp': int((predictions_df['category'] == 'FP').sum()),
            'fn': int((predictions_df['category'] == 'FN').sum()),
            'accuracy': float((predictions_df['y_true'] == predictions_df['y_pred']).mean()),
            'precision': precision,
        },
        'confidence_statistics': {
            'mean_confidence_correct': float(predictions_df[predictions_df['y_true']==predictions_df['y_pred']]['p_defective'].mean()),
            'std_confidence_correct': float(predictions_df[predictions_df['y_true']==predictions_df['y_pred']]['p_defective'].std()),
        },
        'temporal_statistics': {
            'total_sequences': len(sequences),
            'sequences_with_defects': sum(s['has_defects'] for s in sequences),
            'avg_sequence_length': float(np.mean([s['length'] for s in sequences])),
        },
        'postprocessing_impact': {
            'multiscale': {
                'changed_predictions': int(comparison_df['changed_multiscale'].sum()),
                'improvements': int((comparison_df['change_type_multiscale'] == 'Wrong → Correct').sum()),
            },
            'conf_weighted_iso': {
                'changed_predictions': int(comparison_df['changed_conf_iso'].sum()),
                'improvements': int((comparison_df['change_type_conf_iso'] == 'Wrong → Correct').sum()),
            }
        }
    }
    
    def to_serializable(obj):
        if isinstance(obj, (np.integer, np.int32, np.int64)):
            return int(obj)
        elif isinstance(obj, (np.floating, np.float32, np.float64)):
            return float(obj)
        elif isinstance(obj, np.ndarray):
            return obj.tolist()
        return str(obj)
    
    with open(output_dir / "xai_report.json", "w") as f:
        json.dump(report, f, indent=2, default=to_serializable)
    
    print(f"  ✓ Saved XAI report")
    
    print("\n" + "-"*10)
    print("XAI ANALYSIS SUMMARY")
    print("-"*10)
    print(f"\nDataset: {report['dataset_statistics']['total_samples']} samples")
    print(f"Performance: TP={report['performance_summary']['tp']}, "
          f"TN={report['performance_summary']['tn']}, "
          f"FP={report['performance_summary']['fp']}, "
          f"FN={report['performance_summary']['fn']}")
    print(f" Post-Processing Improvements: "
          f"Multiscale={report['postprocessing_impact']['multiscale']['improvements']}, "
          f"Conf+Iso={report['postprocessing_impact']['conf_weighted_iso']['improvements']}")
    
    return report

# MODEL LOADING

def load_model(model_path: Path, device: str = 'cuda') -> nn.Module:
    """Load trained DeiT model."""
    print_section("Loading Model")
    
    model = timm.create_model("deit_small_patch16_224", pretrained=False, num_classes=2)
    state_dict = torch.load(model_path, map_location=device)
    model.load_state_dict(state_dict)
    model.to(device)
    model.eval()
    
    print(f"✓ Loaded: {model_path}")
    print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")
    
    return model

# MAIN EXPERIMENT 6

def run_experiment_6():
    """Main Experiment 6: XAI Analysis."""
    
    print("\n" + "-"*10)
    print("  EXPERIMENT 6: EXPLAINABLE AI ANALYSIS")
    print("-"*10)
    
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"\nDevice: {device}")
    print(f"Output: {OUTPUT_DIR}")
    
    # Check inputs
    if not MODEL_PATH.exists():
        raise FileNotFoundError(f"Model not found: {MODEL_PATH}")
    if not TEST_PREDICTIONS.exists():
        raise FileNotFoundError(f"Predictions not found: {TEST_PREDICTIONS}")
    if not TEST_IMAGES_DIR.exists():
        raise FileNotFoundError(f"Images not found: {TEST_IMAGES_DIR}")
    
    # Load model
    model = load_model(MODEL_PATH, device)
    
    # Load predictions
    print_section("Loading Predictions")
    pred_baseline = pd.read_csv(TEST_PREDICTIONS)
    print(f"  ✓ Baseline: {len(pred_baseline)} samples")
    
    pred_multiscale = pd.read_csv(TEST_PREDICTIONS_MULTISCALE)
    print(f"  ✓ Multiscale: {len(pred_multiscale)} samples")
    
    pred_conf_iso = pd.read_csv(TEST_PREDICTIONS_CONF_ISO)
    print(f"  ✓ Conf+Iso: {len(pred_conf_iso)} samples")
    
    # 1. Error Analysis
    predictions_df = perform_error_analysis(pred_baseline, TEST_IMAGES_DIR)
    create_error_analysis_table(predictions_df, OUTPUT_DIR)
    
    # 2. XAI Visualizations (MAIN PART)
    create_thesis_visualizations(
        predictions_df=predictions_df,
        model=model,
        test_images_dir=TEST_IMAGES_DIR,
        output_dir=OUTPUT_DIR,
        device=device,
        n_per_category=5  # 5 examples each
    )
    
    # 3. Temporal Analysis
    sequences = analyze_temporal_sequences(predictions_df, OUTPUT_DIR)
    
    # 4. Confidence Analysis
    analyze_confidence_distribution(predictions_df, OUTPUT_DIR)
    
    # 5. Post-Processing Impact
    comparison_df = compare_postprocessing_impact(pred_baseline, pred_multiscale, pred_conf_iso, OUTPUT_DIR)
    
    # 6. Generate Report
    report = generate_xai_report(predictions_df, comparison_df, sequences, OUTPUT_DIR)
    
    # Summary
    print("\n" + "-"*10)
    print("EXPERIMENT 6 COMPLETE!")
    print("-"*10)
    print(f"\nOutput: {OUTPUT_DIR}/")
    print("\nGenerated Files:")
    print("  XAI Visualizations:")
    print("    - gradient_saliency/  (simple, fast)")
    print("    - integrated_gradients/  (robust)")
    print("    - smoothgrad/  (noise-reduced)")
    print("    - comparisons/  (side-by-side)")
    print("\n  Analysis:")
    print("    - error_analysis_*.csv")
    print("    - confidence_distribution_analysis.png")
    print("    - sequence_temporal_analysis.png")
    print("    - postprocessing_impact.png")
    print("    - postprocessing_comparison_detailed.csv")
    print("    - xai_report.json")
    
    
    return report

# RUN 

if __name__ == "__main__":
   
    warnings.filterwarnings('ignore')
    
    try:
        report = run_experiment_6()
        print("\n" + "-"*10)
        print("  All analyses complete.")
        print("-"*10)
    except Exception as e:
        print(f"\n ERROR: {str(e)}")
        
        traceback.print_exc()
        raise